# Phase 3A: Occupancy Forecasting Models
## Building 59 South Wing — Stage 1 Forecasting ($t \to t+1$)

This notebook implements, benchmarks, and analyzes candidate machine learning models for:
1. **Primary Task**: Binary Occupancy Classification (`is_occupied_next_hour`)
2. **Secondary Task**: Continuous Headcount Regression (`occ_total_mean_next_hour`)

### Methodology & Constraints:
- Chronological train/validation/test partitions are strictly observed.
- Predictor space contains strictly causal observations available at time $t$.
- Zero energy submeter features are permitted in the occupancy predictor matrix.
- Threshold optimization is performed exclusively on the validation split.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on path
PROJ_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from src.occupancy_model import (
    load_and_split_occupancy_data,
    train_and_eval_classifiers,
    train_and_eval_headcount_regressors,
    load_occupancy_model_artifacts
)
from src.evaluation import (
    compute_classification_metrics,
    plot_confusion_matrix,
    plot_roc_pr_curves,
    plot_feature_importance
)

DATA_PATH = PROJ_ROOT / 'data' / 'processed' / 'joint_modeling_data.parquet'
MODEL_DIR = PROJ_ROOT / 'models' / 'occupancy'


## 1. Load Data & Verify Feature Space

In [2]:
occ_data = load_and_split_occupancy_data(DATA_PATH)

print(f"Total Predictors: {len(occ_data['feature_cols'])}")
print(f"Train samples   : {len(occ_data['train']['X'])}")
print(f"Val samples     : {len(occ_data['val']['X'])}")
print(f"Test samples    : {len(occ_data['test']['X'])}")

# Confirm zero energy submeters in predictor space
energy_leaks = [c for c in occ_data['feature_cols'] if 'kwh' in c or 'lig' in c or 'mels' in c or 'hvac' in c]
print(f"Energy features in occupancy predictor matrix: {energy_leaks} (Must be empty)")
assert len(energy_leaks) == 0, "Energy leakage detected in occupancy model!"


Total Predictors: 34
Train samples   : 4063
Val samples     : 888
Test samples    : 994
Energy features in occupancy predictor matrix: [] (Must be empty)


## 2. Train & Benchmark Occupancy Classifiers

In [3]:
cls_results = train_and_eval_classifiers(
    occ_data['train']['X'],
    occ_data['train']['y_cls'],
    occ_data['val']['X'],
    occ_data['val']['y_cls'],
    occ_data['feature_cols'],
    random_state=42
)

summary_rows = []
for name, r in cls_results.items():
    summary_rows.append({
        'Model': name,
        'Precision': r['metrics_tuned']['precision'],
        'Recall': r['metrics_tuned']['recall'],
        'F1 (Tuned)': r['metrics_tuned']['f1'],
        'F1 (0.5)': r['metrics_default']['f1'],
        'PR-AUC': r['metrics_default']['pr_auc'],
        'ROC-AUC': r['metrics_default']['roc_auc'],
        'Bal Acc': r['metrics_tuned']['balanced_accuracy'],
        'Accuracy': r['metrics_tuned']['accuracy'],
        'Optimal Threshold': r['best_threshold']
    })

df_cls_val = pd.DataFrame(summary_rows).sort_values(by='F1 (Tuned)', ascending=False)
display(df_cls_val)


[DEPENDENCY WARNING] LightGBM is unavailable or failed runtime execution: exception: access violation reading 0x0000000000000000. Skipping and continuing with remaining models.


,Model,Precision,Recall,F1 (Tuned),F1 (0.5),PR-AUC,ROC-AUC,Bal Acc,Accuracy,Optimal Threshold
2,Random Forest,0.9294,0.9461,0.9377,0.9365,0.9815,0.9433,0.8640,0.9054,0.51
3,XGBoost,0.9247,0.9371,0.9309,0.9269,0.9818,0.9437,0.8527,0.8953,0.48
1,Decision Tree,0.9297,0.9311,0.9304,0.9164,0.9765,0.9400,0.8588,0.8953,0.29
0,Logistic Regression (L2),0.8619,0.9626,0.9095,0.8581,0.9400,0.8673,0.7472,0.8559,0.24


## 3. Train Headcount Regressors (Secondary Task)

In [4]:
hc_results = train_and_eval_headcount_regressors(
    occ_data['train']['X'],
    occ_data['train']['y_reg'],
    occ_data['val']['X'],
    occ_data['val']['y_reg'],
    random_state=42
)

hc_rows = []
for name, r in hc_results.items():
    hc_rows.append({
        'Model': name,
        'Raw MAE': r['raw_metrics']['mae'],
        'Raw RMSE': r['raw_metrics']['rmse'],
        'Raw R2': r['raw_metrics']['r2'],
        'Log1p MAE': r['log_metrics']['mae'],
        'Log1p RMSE': r['log_metrics']['rmse'],
        'Log1p R2': r['log_metrics']['r2'],
        'Preferred': r['preferred_regime']
    })

df_hc_val = pd.DataFrame(hc_rows).sort_values(by='Raw RMSE')
display(df_hc_val)


[DEPENDENCY WARNING] LightGBM is unavailable or failed runtime execution: exception: access violation reading 0x0000000000000000. Skipping and continuing with remaining models.


,Model,Raw MAE,Raw RMSE,Raw R2,Log1p MAE,Log1p RMSE,Log1p R2,Preferred
2,XGBoost,3.7774,6.1781,0.7739,3.4722,7.0019,0.7096,raw
1,Random Forest,3.4362,6.2087,0.7717,3.5807,6.8556,0.7216,raw
0,Ridge Regression,5.9381,8.1913,0.6026,5.7050,13.2818,-0.0448,raw


## 4. Final Evaluation of Champion on Test Split

In [5]:
# Load Champion Model Artifacts
champion_artifact = load_occupancy_model_artifacts(MODEL_DIR)
model = champion_artifact['model']
scaler = champion_artifact['scaler']
threshold = champion_artifact['threshold']

print(f"Loaded Champion Model: {champion_artifact['metadata']['model_name']}")
print(f"Frozen Optimal Threshold: {threshold}")

# Evaluate on Test Split
X_test_in = scaler.transform(occ_data['test']['X']) if scaler else occ_data['test']['X'].values
test_probs = model.predict_proba(X_test_in)[:, 1]
test_preds = (test_probs >= threshold).astype(int)

test_metrics = compute_classification_metrics(occ_data['test']['y_cls'], test_preds, test_probs)
for k, v in test_metrics.items():
    if k != 'confusion_matrix':
        print(f"{k:20s}: {v}")
print(f"Confusion Matrix    : {test_metrics['confusion_matrix']}")


Loaded Champion Model: Random Forest
Frozen Optimal Threshold: 0.51
accuracy            : 0.8712
balanced_accuracy   : 0.8032
precision           : 0.9018
recall              : 0.9327
f1                  : 0.917
roc_auc             : 0.9364
pr_auc              : 0.9805
Confusion Matrix    : [[159, 77], [51, 707]]


## 5. Visual Diagnostics

In [6]:
# Confusion Matrix Heatmap
plot_confusion_matrix(test_metrics['confusion_matrix'], title=f"{champion_artifact['metadata']['model_name']} (Test Split)")
plt.show()

# ROC & PR Curves
plot_roc_pr_curves(occ_data['test']['y_cls'], test_probs, title_prefix=champion_artifact['metadata']['model_name'])
plt.show()

# Feature Importance
importances = getattr(model, 'feature_importances_', None)
if importances is not None:
    plot_feature_importance(occ_data['feature_cols'], importances, top_n=15, title='Random Forest Feature Importance')
    plt.show()


C:\Users\thund\OneDrive\Desktop\ML PROJECT\src\evaluation.py:247: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_imp, x='Importance', y='Feature', palette='Blues_r', ax=ax)
